# Notebook 04 (v1.1) — Évaluation cross-dataset (CIC-IDS2017 ⇄ UNSW-NB15)

**Question :** un modèle entraîné sur un dataset conserve-t-il ses performances sur l'autre ?

**Principe.** Pour chaque dataset source, chaque fold externe, chaque graine et chaque modèle, **un seul modèle** est entraîné puis évalué deux fois avec **le même seuil** :
1. **intra** : sur le test externe du dataset source (partitions v2 réutilisées à l'identique) ;
2. **cross** : sur **tout** l'échantillon du dataset cible, jamais vu à l'entraînement.

**Protocole de seuil (logique v1).** Le seuil est choisi sur le premier split interne groupé (80 % / 20 % du train externe), jamais sur le test externe ni sur la cible. `REFIT_ON_FULL_TRAIN` active une variante de sensibilité (réentraînement sur 100 % du train externe, même seuil).

**Nouveautés de la v1.1** (par rapport à la première version du notebook) :
- étiquettes vraies, indices et scores de validation interne sauvegardés dans chaque fichier de prédictions (analyse des erreurs et ablation du seuil sans réentraînement) ;
- scores enregistrés en `float64` pour que le recalcul des métriques depuis les fichiers soit **exactement** celui du CSV ;
- comptage des avertissements de convergence (tous les ajustements d'une expérience) ;
- contrôle de couverture des folds externes (chaque observation en test exactement une fois) ;
- sauvegarde atomique du CSV, clé de reprise incluant la cible, vérification des fichiers de prédictions ;
- nouveau `TAG` (`..._v1_1`) : les sorties d'une éventuelle exécution de la première version ne sont ni réutilisées ni écrasées.

**Sorties** : `results/tables/cross_dataset_baselines_v1_1.csv` (+ résumé) et `results/predictions/cross_dataset_v1_1/*.npz`.

**Rappel de prudence.** Une baisse cross-dataset ne prouve ni fuite de données ni artefact : décalage de distribution, sémantique des variables et labellisation peuvent l'expliquer.

## 1. Configuration

In [1]:
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, average_precision_score, roc_auc_score,
    confusion_matrix, precision_recall_curve)
from sklearn.exceptions import ConvergenceWarning

SEEDS = [42, 123, 2026]
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5
MODELS_TO_RUN = ["Logistic Regression", "Random Forest", "MLP"]  # MLP en dernier : le plus lent
REFIT_ON_FULL_TRAIN = False   # False = comparable à v1 ; True = variante de sensibilité
RUN_FULL_EXPERIMENT = True    # False = pilote uniquement
SAVE_PREDICTIONS = True

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT_DIR / "data" / "interim"
TABLES_DIR = ROOT_DIR / "results" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = ["flow_duration_s", "fwd_packets", "bwd_packets", "fwd_bytes", "bwd_bytes"]
TARGET_COLUMN = "target"
SHORT = {"CIC-IDS2017": "CIC", "UNSW-NB15": "UNSW"}

VERSION = "v1_1"  # nouvelle version : ne mélange pas les sorties de la première version du notebook
TAG = f"cross_dataset_baselines_{VERSION}_refit" if REFIT_ON_FULL_TRAIN else f"cross_dataset_baselines_{VERSION}"
RESULTS_FILE = TABLES_DIR / f"{TAG}.csv"
SUMMARY_FILE = TABLES_DIR / f"{TAG.replace('baselines', 'baselines_summary')}.csv"
PRED_DIR = ROOT_DIR / "results" / "predictions" / TAG.replace("_baselines", "")
if SAVE_PREDICTIONS:
    PRED_DIR.mkdir(parents=True, exist_ok=True)

def pred_filename(source_short, target_short, model_name, fold_number, seed):
    return f"{source_short}_to_{target_short}_{model_name.replace(' ', '_')}_f{fold_number}_s{seed}.npz"

print("Racine      :", ROOT_DIR)
print("Sortie      :", RESULTS_FILE)
print("Prédictions :", PRED_DIR if SAVE_PREDICTIONS else "désactivées")
print("Refit complet :", REFIT_ON_FULL_TRAIN, "| Seeds :", SEEDS, "| Modèles :", MODELS_TO_RUN)

Racine      : D:\Master-SDIA\CYB-1-IDS-Generalization
Sortie      : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables\cross_dataset_baselines_v1_1.csv
Prédictions : D:\Master-SDIA\CYB-1-IDS-Generalization\results\predictions\cross_dataset_v1_1
Refit complet : False | Seeds : [42, 123, 2026] | Modèles : ['Logistic Regression', 'Random Forest', 'MLP']


## 2. Chargement des deux datasets et contrôles
Mêmes données et même règle de groupes que v1/v2 (combinaisons exactes des cinq variables ; ce ne sont pas des sessions). Contrôles ajoutés : variables numériques, clés `.npz` attendues, **chaque observation en test exactement une fois sur les cinq folds**, train = complément du test.

In [2]:
def load_dataset(dataset_name, short_name, sample_filename, splits_filename):
    sample_path = DATA_DIR / sample_filename
    splits_path = TABLES_DIR / splits_filename
    if not sample_path.exists(): raise FileNotFoundError(f"Échantillon introuvable : {sample_path}")
    if not splits_path.exists(): raise FileNotFoundError(f"Folds introuvables : {splits_path}")
    df = pd.read_csv(sample_path)
    missing = [c for c in FEATURE_COLUMNS + [TARGET_COLUMN] if c not in df.columns]
    if missing: raise ValueError(f"{dataset_name}: colonnes manquantes {missing}")
    X = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="raise")
    y = df[TARGET_COLUMN].astype(int).to_numpy()
    if set(np.unique(y)) != {0, 1}: raise ValueError(f"{dataset_name}: target doit contenir 0 et 1")
    if not np.isfinite(X.to_numpy(dtype=float)).all(): raise ValueError(f"{dataset_name}: valeurs non finies")
    groups, _ = pd.factorize(pd.MultiIndex.from_frame(X), sort=False)
    groups = np.asarray(groups, dtype=np.int64)

    with np.load(splits_path) as z:
        expected_keys = {f"{p}_{i}" for i in range(1, N_OUTER_FOLDS + 1) for p in ("train", "test")}
        missing_keys = expected_keys - set(z.files)
        if missing_keys: raise KeyError(f"{dataset_name}: clés absentes {sorted(missing_keys)}; disponibles: {z.files}")
        extra_keys = set(z.files) - expected_keys
        if extra_keys: print(f"  (info) clés supplémentaires ignorées dans {splits_filename}: {sorted(extra_keys)}")
        outer_splits = [(z[f"train_{i}"].astype(np.int64).copy(), z[f"test_{i}"].astype(np.int64).copy())
                        for i in range(1, N_OUTER_FOLDS + 1)]

    n = len(X)
    for fold, (tr, te) in enumerate(outer_splits, 1):
        if len(np.unique(tr)) != len(tr) or len(np.unique(te)) != len(te): raise AssertionError(f"Indices dupliqués fold {fold}")
        if min(tr.min(), te.min()) < 0 or max(tr.max(), te.max()) >= n: raise AssertionError(f"Indice hors limites fold {fold}")
        if np.intersect1d(tr, te).size: raise AssertionError(f"Chevauchement indices fold {fold}")
        if len(tr) + len(te) != n: raise AssertionError(f"Fold {fold}: train + test != ensemble complet")
        if np.intersect1d(np.unique(groups[tr]), np.unique(groups[te])).size: raise AssertionError(f"Fuite groupes fold {fold}")
    all_test = np.concatenate([te for _, te in outer_splits])
    if not np.array_equal(np.sort(all_test), np.arange(n)):
        raise AssertionError(f"{dataset_name}: les tests externes ne couvrent pas chaque observation exactement une fois")

    print(f"{dataset_name}: X={X.shape}, groupes={len(np.unique(groups))}, classes={dict(zip(*np.unique(y, return_counts=True)))}")
    return {"name": dataset_name, "short": short_name, "X": X, "y": y, "groups": groups, "outer_splits": outer_splits}

cic_data = load_dataset("CIC-IDS2017", "CIC", "cic_harmonized_sample_v2.csv", "cic_group_splits_v2.npz")
unsw_data = load_dataset("UNSW-NB15", "UNSW", "unsw_harmonized_sample_v2.csv", "unsw_group_splits_v2.npz")

CIC-IDS2017: X=(100000, 5), groupes=74294, classes={np.int64(0): np.int64(80212), np.int64(1): np.int64(19788)}
UNSW-NB15: X=(100000, 5), groupes=64117, classes={np.int64(0): np.int64(87423), np.int64(1): np.int64(12577)}


## 3. Modèles, seuil et métriques
Hyperparamètres **identiques à v1/v2**. Le scaler est ajusté dans le pipeline, uniquement sur les données d'entraînement du modèle.

In [3]:
def build_models(seed):
    return {
        "Logistic Regression": Pipeline([("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=seed))]),
        "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
            random_state=seed, n_jobs=-1),
        "MLP": Pipeline([("scaler", StandardScaler()),
            ("model", MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
                max_iter=300, early_stopping=False, n_iter_no_change=15, random_state=seed))]),
    }

def select_threshold(y_true, probabilities):
    # Seuil maximisant le F1 ; refuse les entrées invalides au lieu d'un seuil de secours silencieux.
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    if len(y_true) != len(probabilities): raise ValueError("Tailles incompatibles")
    if not np.isfinite(probabilities).all(): raise ValueError("Probabilités non finies")
    if len(np.unique(y_true)) < 2: raise ValueError("Les deux classes doivent être présentes")
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    if len(thresholds) == 0: raise ValueError("Aucun seuil disponible")
    den = precision[:-1] + recall[:-1]
    f1_values = np.divide(2 * precision[:-1] * recall[:-1], den, out=np.zeros_like(den), where=den != 0)
    best = int(np.nanargmax(f1_values))
    return float(thresholds[best]), float(f1_values[best])

def evaluate(y_true, scores, threshold, prefix):
    # Métriques au seuil fixé SUR LA SOURCE. Les colonnes *_diag_* sont purement diagnostiques
    # (meilleur F1 possible avec les labels de l'ensemble évalué) : jamais une performance de détecteur.
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    oracle_thr, oracle_f1 = select_threshold(y_true, scores)
    return {
        f"{prefix}_f1": f1_score(y_true, pred, zero_division=0),
        f"{prefix}_pr_auc": average_precision_score(y_true, scores),
        f"{prefix}_roc_auc": roc_auc_score(y_true, scores),
        f"{prefix}_precision": precision_score(y_true, pred, zero_division=0),
        f"{prefix}_recall": recall_score(y_true, pred, zero_division=0),
        f"{prefix}_accuracy": accuracy_score(y_true, pred),
        f"{prefix}_balanced_accuracy": balanced_accuracy_score(y_true, pred),
        f"{prefix}_prevalence": float(np.mean(y_true)),
        f"{prefix}_pred_positive_rate": float(np.mean(pred)),
        f"{prefix}_tn": int(tn), f"{prefix}_fp": int(fp), f"{prefix}_fn": int(fn), f"{prefix}_tp": int(tp),
        f"{prefix}_diag_oracle_f1": oracle_f1, f"{prefix}_diag_oracle_threshold": oracle_thr,
    }

def fit_recording_warnings(model, X_fit, y_fit):
    # Ajuste le modèle et compte les ConvergenceWarning ; les autres avertissements sont affichés.
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(X_fit, y_fit)
    for w in caught:
        if not issubclass(w.category, ConvergenceWarning):
            print("Avertissement :", w.message)
    return sum(issubclass(w.category, ConvergenceWarning) for w in caught)

print("Modèles disponibles :", list(build_models(0).keys()))

Modèles disponibles : ['Logistic Regression', 'Random Forest', 'MLP']


## 4. Une expérience source → cible
1. Premier split interne groupé (`StratifiedGroupKFold`, graine de l'expérience) sur le train externe ; contrôles de non-chevauchement des indices **absolus** et des groupes (entre eux et avec le test externe).
2. Entraînement sur la partie interne d'entraînement ; seuil choisi sur la partie interne de validation.
3. (Option) réentraînement sur tout le train externe.
4. Évaluation intra (test externe source) et cross (tout le dataset cible) avec le même seuil.
5. Sauvegarde des scores et des étiquettes vraies. Les scores de validation interne viennent du modèle utilisé pour choisir le seuil (avec `REFIT_ON_FULL_TRAIN=True`, ce n'est pas le modèle évalué).

In [4]:
def run_one_experiment(source, target, model_name, fold_number, seed):
    X, y, groups = source["X"], source["y"], source["groups"]
    train_idx, test_idx = source["outer_splits"][fold_number - 1]
    X_tr, y_tr, g_tr = X.iloc[train_idx], y[train_idx], groups[train_idx]

    cv = StratifiedGroupKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=seed)
    inner_tr, inner_va = next(iter(cv.split(X_tr, y_tr, g_tr)))
    inner_train_idx, inner_val_idx = train_idx[inner_tr], train_idx[inner_va]   # indices absolus
    if np.intersect1d(inner_train_idx, inner_val_idx).size: raise AssertionError("Chevauchement indices split interne")
    if not (np.isin(inner_train_idx, train_idx).all() and np.isin(inner_val_idx, train_idx).all()):
        raise AssertionError("Le split interne sort du train externe")
    if np.intersect1d(np.concatenate([inner_train_idx, inner_val_idx]), test_idx).size:
        raise AssertionError("Le split interne touche le test externe")
    if np.intersect1d(np.unique(groups[inner_train_idx]), np.unique(groups[inner_val_idx])).size:
        raise AssertionError("Fuite groupes split interne")

    model = build_models(seed)[model_name]
    start = time.perf_counter()
    n_warn = fit_recording_warnings(model, X.iloc[inner_train_idx], y[inner_train_idx])
    val_scores = model.predict_proba(X.iloc[inner_val_idx])[:, 1]
    threshold, val_f1 = select_threshold(y[inner_val_idx], val_scores)

    final_model = model
    if REFIT_ON_FULL_TRAIN:
        final_model = build_models(seed)[model_name]
        n_warn += fit_recording_warnings(final_model, X_tr, y_tr)
    train_time = time.perf_counter() - start

    intra_scores = final_model.predict_proba(X.iloc[test_idx])[:, 1]
    cross_scores = final_model.predict_proba(target["X"])[:, 1]

    mlp_n_iter, mlp_hit_max = np.nan, np.nan
    if model_name == "MLP":
        est = final_model.named_steps["model"]
        mlp_n_iter = int(est.n_iter_)
        mlp_hit_max = bool(est.n_iter_ >= est.max_iter)

    if SAVE_PREDICTIONS:
        fname = pred_filename(source["short"], target["short"], model_name, fold_number, seed)
        np.savez_compressed(PRED_DIR / fname,
            threshold=np.float64(threshold),
            source_test_idx=test_idx.astype(np.int64),
            source_test_y_true=y[test_idx].astype(np.int8),
            source_test_score=intra_scores.astype(np.float64),     # float64 : recalcul exact des métriques
            target_y_true=target["y"].astype(np.int8),
            target_score=cross_scores.astype(np.float64),
            inner_val_idx=inner_val_idx.astype(np.int64),
            inner_val_y_true=y[inner_val_idx].astype(np.int8),
            inner_val_score=val_scores.astype(np.float64))

    row = {
        "source": source["name"], "target": target["name"], "model": model_name,
        "fold": fold_number, "seed": seed, "refit_full_train": REFIT_ON_FULL_TRAIN,
        "n_train_outer": len(train_idx), "n_inner_train": len(inner_train_idx), "n_inner_val": len(inner_val_idx),
        "n_source_test": len(test_idx), "n_target": len(target["X"]),
        "threshold": threshold, "inner_val_f1": val_f1, "training_time_s": train_time,
        "n_convergence_warnings": int(n_warn), "mlp_n_iter": mlp_n_iter, "mlp_hit_max_iter": mlp_hit_max,
    }
    row.update(evaluate(y[test_idx], intra_scores, threshold, "intra"))
    row.update(evaluate(target["y"], cross_scores, threshold, "cross"))
    return row
print("run_one_experiment définie.")

run_one_experiment définie.


## 5. Essai pilote et contrôle du fichier de prédictions
CIC → UNSW, Logistic Regression, fold 1, seed 42. La cellule recharge le `.npz` et **recalcule F1 intra et cross depuis les scores sauvegardés** : si les deux valeurs coïncident avec celles du pilote, les fichiers suffisent bien pour l'ablation du seuil et l'analyse des erreurs.

In [5]:
pilot = run_one_experiment(cic_data, unsw_data, "Logistic Regression", 1, 42)
display(pd.DataFrame([pilot]).T.rename(columns={0: "Valeur"}))
print(f"Intra F1 = {pilot['intra_f1']:.4f} | Cross F1 = {pilot['cross_f1']:.4f} | "
      f"Intra PR-AUC = {pilot['intra_pr_auc']:.4f} | Cross PR-AUC = {pilot['cross_pr_auc']:.4f} "
      f"(prévalence cible = {pilot['cross_prevalence']:.4f})")

if SAVE_PREDICTIONS:
    pilot_file = PRED_DIR / pred_filename("CIC", "UNSW", "Logistic Regression", 1, 42)
    assert pilot_file.exists(), f"Fichier pilote absent : {pilot_file}"
    with np.load(pilot_file) as d:
        print("\nClés du .npz :", d.files)
        for k in d.files: print(f"  {k}: {d[k].shape} {d[k].dtype}")
        required = {"threshold", "source_test_idx", "source_test_y_true", "source_test_score",
                    "target_y_true", "target_score", "inner_val_idx", "inner_val_y_true", "inner_val_score"}
        assert required <= set(d.files), f"Clés manquantes : {required - set(d.files)}"
        assert np.array_equal(d["source_test_y_true"], cic_data["y"][d["source_test_idx"]]), "y_true source incohérent"
        assert np.array_equal(d["target_y_true"], unsw_data["y"]), "y_true cible incohérent"
        assert np.array_equal(d["inner_val_y_true"], cic_data["y"][d["inner_val_idx"]]), "y_true validation incohérent"
        thr = float(d["threshold"])
        f1_intra = f1_score(d["source_test_y_true"], (d["source_test_score"] >= thr).astype(int), zero_division=0)
        f1_cross = f1_score(d["target_y_true"], (d["target_score"] >= thr).astype(int), zero_division=0)
    assert abs(f1_intra - pilot["intra_f1"]) < 1e-12 and abs(f1_cross - pilot["cross_f1"]) < 1e-12, "Recalcul différent du pilote"
    print("\nVérification du pilote réussie : métriques recalculées depuis le .npz identiques.")

,Valeur
source,CIC-IDS2017
target,UNSW-NB15
model,Logistic Regression
fold,1
seed,42
refit_full_train,False
n_train_outer,79999
n_inner_train,64000
n_inner_val,15999
n_source_test,20001


Intra F1 = 0.5307 | Cross F1 = 0.0005 | Intra PR-AUC = 0.4837 | Cross PR-AUC = 0.6470 (prévalence cible = 0.1258)

Clés du .npz : ['threshold', 'source_test_idx', 'source_test_y_true', 'source_test_score', 'target_y_true', 'target_score', 'inner_val_idx', 'inner_val_y_true', 'inner_val_score']
  threshold: () float64
  source_test_idx: (20001,) int64
  source_test_y_true: (20001,) int8
  source_test_score: (20001,) float64
  target_y_true: (100000,) int8
  target_score: (100000,) float64
  inner_val_idx: (15999,) int64
  inner_val_y_true: (15999,) int8
  inner_val_score: (15999,) float64

Vérification du pilote réussie : métriques recalculées depuis le .npz identiques.


## 6. Expérience complète avec sauvegarde atomique et reprise
Boucle : modèles (le plus lent en dernier) → source → fold → graine. La clé de reprise inclut la direction (source, cible). Le CSV est écrit dans un fichier temporaire puis remplacé : une interruption ne peut pas le corrompre.

In [6]:
KEY_COLUMNS = ["source", "target", "model", "fold", "seed"]
REQUIRED_COLUMNS = set(KEY_COLUMNS) | {"intra_f1", "cross_f1", "intra_pr_auc", "cross_pr_auc", "n_convergence_warnings"}

def save_results_safely(df, output_file):
    temp_file = output_file.with_name(output_file.stem + "_temporary.csv")
    df.to_csv(temp_file, index=False)
    temp_file.replace(output_file)

if RESULTS_FILE.exists():
    results_df = pd.read_csv(RESULTS_FILE)
    missing_columns = REQUIRED_COLUMNS - set(results_df.columns)
    if missing_columns: raise ValueError(f"Fichier de résultats incompatible, colonnes manquantes : {missing_columns}")
    if results_df.duplicated(subset=KEY_COLUMNS).any(): raise ValueError("Le CSV contient des expériences dupliquées.")
    completed = set(zip(results_df.source, results_df.target, results_df.model,
                        results_df.fold.astype(int), results_df.seed.astype(int)))
    print(f"Reprise : {len(completed)} expériences déjà enregistrées.")
else:
    results_df, completed = pd.DataFrame(), set()

directions = [(cic_data, unsw_data), (unsw_data, cic_data)]
if RUN_FULL_EXPERIMENT:
    total = len(directions) * len(MODELS_TO_RUN) * N_OUTER_FOLDS * len(SEEDS)
    print(f"Expériences terminées : {len(completed)}/{total}")
    for model_name in MODELS_TO_RUN:
        for source, target in directions:
            for fold_number in range(1, N_OUTER_FOLDS + 1):
                for seed in SEEDS:
                    key = (source["name"], target["name"], model_name, fold_number, seed)
                    if key in completed: continue
                    print(f"{source['name']} -> {target['name']} | {model_name} | fold={fold_number} | seed={seed}")
                    row = run_one_experiment(source, target, model_name, fold_number, seed)
                    results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
                    save_results_safely(results_df, RESULTS_FILE)
                    completed.add(key)
    print("Terminé :", RESULTS_FILE)
else:
    print("RUN_FULL_EXPERIMENT=False : pilote uniquement.")

Expériences terminées : 0/90
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=1 | seed=42
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=1 | seed=123
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=1 | seed=2026
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=2 | seed=42
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=2 | seed=123
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=2 | seed=2026
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=3 | seed=42
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=3 | seed=123
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=3 | seed=2026
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=4 | seed=42
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=4 | seed=123
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=4 | seed=2026
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=5 | seed=42
CIC-IDS2017 -> UNSW-NB15 | Logistic Regression | fold=5 | seed=123
CIC-IDS2017 -> UNSW-NB15 | Logisti

## 7. Vérification de complétude
Combinaisons attendues, doublons, valeurs manquantes (hors colonnes MLP) et **présence d'un fichier de prédictions pour chaque ligne du CSV**.

In [7]:
if RESULTS_FILE.exists():
    results_df = pd.read_csv(RESULTS_FILE)
    pairs = [("CIC-IDS2017", "UNSW-NB15"), ("UNSW-NB15", "CIC-IDS2017")]
    expected = {(s, t, m, f, sd) for s, t in pairs for m in MODELS_TO_RUN
                for f in range(1, N_OUTER_FOLDS + 1) for sd in SEEDS}
    observed = set(zip(results_df.source, results_df.target, results_df.model,
                       results_df.fold.astype(int), results_df.seed.astype(int)))
    duplicates = int(results_df.duplicated(subset=KEY_COLUMNS).sum())
    print("Attendus :", len(expected), "| obtenus :", len(observed), "| manquants :", len(expected - observed),
          "| inattendus :", len(observed - expected), "| doublons :", duplicates)
    print("Valeurs manquantes (hors colonnes MLP) :",
          int(results_df.drop(columns=["mlp_n_iter", "mlp_hit_max_iter"]).isna().sum().sum()))
    if SAVE_PREDICTIONS:
        absent = [r for r in results_df.itertuples()
                  if not (PRED_DIR / pred_filename(SHORT[r.source], SHORT[r.target], r.model, int(r.fold), int(r.seed))).exists()]
        print("Fichiers de prédictions absents :", len(absent))
        if absent: print("Exemple :", absent[0].source, absent[0].model, absent[0].fold, absent[0].seed)
    if duplicates: raise AssertionError("Combinaisons expérimentales dupliquées")
else:
    print("Fichier de résultats absent.")

Attendus : 90 | obtenus : 90 | manquants : 0 | inattendus : 0 | doublons : 0
Valeurs manquantes (hors colonnes MLP) : 0
Fichiers de prédictions absents : 0


## 8. Résumé : intra, cross et écart apparié
`delta` = intra − cross, calculé expérience par expérience (même modèle, même seuil). L'écart-type décrit la variabilité observée ; ce n'est pas un intervalle de confiance : les folds ne sont pas des répétitions indépendantes, et **les mêmes observations cibles sont réutilisées dans chaque expérience cross**.

In [8]:
if RESULTS_FILE.exists():
    df = pd.read_csv(RESULTS_FILE)
    df["delta_f1"] = df["intra_f1"] - df["cross_f1"]
    df["delta_pr_auc"] = df["intra_pr_auc"] - df["cross_pr_auc"]
    metrics = ["intra_f1", "cross_f1", "delta_f1",
               "intra_pr_auc", "cross_pr_auc", "delta_pr_auc",
               "intra_roc_auc", "cross_roc_auc",
               "intra_recall", "cross_recall", "intra_precision", "cross_precision",
               "cross_prevalence", "threshold",
               "intra_diag_oracle_f1", "cross_diag_oracle_f1"]
    summary = df.groupby(["source", "target", "model"])[metrics].agg(["mean", "std"]).round(4)
    pd.set_option("display.width", 250)
    display(summary)
    summary.to_csv(SUMMARY_FILE)
    print("Synthèse :", SUMMARY_FILE)
    mlp = df[df.model == "MLP"]
    if len(mlp):
        print(f"\nMLP : {int(mlp.mlp_hit_max_iter.astype(bool).sum())}/{len(mlp)} modèles évalués ont atteint max_iter ; "
              f"{int((mlp.n_convergence_warnings > 0).sum())}/{len(mlp)} expériences ont émis au moins un avertissement de convergence.")
else:
    print("Exécuter d'abord l'expérience complète.")

intra_f1         cross_f1         delta_f1         intra_pr_auc         cross_pr_auc          ... cross_precision         cross_prevalence      threshold         intra_diag_oracle_f1          \
                                                mean     std     mean     std     mean     std         mean     std         mean     std  ...            mean     std             mean  std      mean     std                 mean     std   
source      target      model                                                                                                             ...                                                                                                
CIC-IDS2017 UNSW-NB15   Logistic Regression   0.6062  0.0350   0.0533  0.2047   0.5529  0.2065       0.5157  0.0402       0.6527  0.0213  ...          0.0725  0.2372           0.1258  0.0    0.4742  0.0029               0.6160  0.0249   
                        MLP                   0.7693  0.0337   0.0038  0.0038   0.7655  0.0343       0.8638  0.0302       0.1338  0.0189  ...          0.2448  0.2461           0.1258  0.0    0.3125  0.1153               0.7868  0.0327   
                        Random Forest         0.9536  0.0237   0.0000  0.0000   0.9536  0.0237       0.9825  0.0097       0.1346  0.0197  ...          0.0000  0.0000           0.1258  0.0    0.6889  0.0517               0.9548  0.0234   
UNSW-NB15   CIC-IDS2017 Logistic Regression   0.4261  0.2216   0.4474  0.0139  -0.0213  0.2330       0.4074  0.0910       0.4153  0.0288  ...          0.3506  0.0616           0.1979  0.0    0.6951  0.0185               0.6422  0.0488   
                        MLP                   0.6066  0.3365   0.1385  0.0571   0.4681  0.3131       0.9439  0.0298       0.1996  0.0256  ...          0.1526  0.0631           0.1979  0.0    0.7381  0.2608               0.9365  0.0216   
                        Random Forest         0.5732  0.3822   0.1008  0.1349   0.4723  0.2844       0.9815  0.0094       0.1871  0.0181  ...          0.0754  0.0936           0.1979  0.0    0.7257  0.3261               0.9617  0.0161   

                                            cross_diag_oracle_f1          
                                                            mean     std  
source      target      model                                             
CIC-IDS2017 UNSW-NB15   Logistic Regression               0.7832  0.0204  
                        MLP                               0.3330  0.0385  
                        Random Forest                     0.2234  0.0000  
UNSW-NB15   CIC-IDS2017 Logistic Regression               0.4770  0.0077  
                        MLP                               0.3755  0.0356  
                        Random Forest                     0.3689  0.0182  

[6 rows x 32 columns]

Synthèse : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables\cross_dataset_baselines_summary_v1_1.csv

MLP : 3/30 modèles évalués ont atteint max_iter ; 3/30 expériences ont émis au moins un avertissement de convergence.


## 9. Contrôle exploratoire de cohérence avec v1 (intra)
Compare le F1 intra (et le seuil, si v1 l'a enregistré) aux résultats v1 sur les mêmes combinaisons. Une coïncidence est un **indice fort** que la procédure est équivalente, pas une preuve : le protocole de seuil, les splits internes et les données d'entraînement doivent aussi être identiques. En cas d'écarts, la comparaison intra/cross de ce notebook reste valide (même exécution) ; il faut seulement le signaler.

In [9]:
V1_FILE = TABLES_DIR / "intra_dataset_baselines_v1.csv"
if V1_FILE.exists() and RESULTS_FILE.exists() and not REFIT_ON_FULL_TRAIN:
    v1 = pd.read_csv(V1_FILE)
    print("Colonnes v1 :", v1.columns.tolist())
    f1_col = next((c for c in ["test_f1", "f1", "f1_score"] if c in v1.columns), None)
    cur = pd.read_csv(RESULTS_FILE).rename(columns={"source": "dataset"})
    keys = ["dataset", "model", "fold", "seed"]
    if f1_col and all(k in v1.columns for k in keys):
        cols_v1 = keys + [f1_col] + (["threshold"] if "threshold" in v1.columns else [])
        merged = v1[cols_v1].merge(cur[keys + ["intra_f1", "threshold"]], on=keys, suffixes=("_v1", "_cur"))
        d_f1 = (merged[f1_col] - merged["intra_f1"]).abs().max()
        print(f"Lignes comparées : {len(merged)} | écart absolu max du F1 : {d_f1:.6f}")
        if "threshold_v1" in merged.columns:
            d_thr = (merged["threshold_v1"] - merged["threshold_cur"]).abs().max()
            print(f"Écart absolu max du seuil : {d_thr:.6f}")
        print("Résultats intra cohérents avec v1 (indice fort d'une procédure équivalente)." if d_f1 < 1e-6
              else "Écarts présents : procédure non strictement identique à v1 (à documenter).")
    else:
        print("Colonnes v1 non reconnues : comparaison ignorée.")
else:
    print("Contrôle ignoré (fichier v1 absent, résultats absents ou mode refit).")

Colonnes v1 : ['dataset', 'model', 'fold', 'seed', 'n_train', 'n_validation', 'n_test', 'threshold', 'validation_f1', 'test_f1', 'test_pr_auc', 'test_precision', 'test_recall', 'test_accuracy', 'test_balanced_accuracy', 'training_time_s', 'tn', 'fp', 'fn', 'tp']
Lignes comparées : 90 | écart absolu max du F1 : 0.000000
Écart absolu max du seuil : 0.000000
Résultats intra cohérents avec v1 (indice fort d'une procédure équivalente).


## 10. Diagnostic de décalage de distribution (descriptif)
Statistique de Kolmogorov-Smirnov par variable entre CIC et UNSW (valeurs en `log1p`) et part des lignes cibles dont le vecteur des cinq variables existe exactement dans l'autre dataset. **Descriptif** : ces mesures aident à interpréter la chute cross-dataset sans en établir la cause.

In [10]:
from scipy.stats import ks_2samp

rows = []
for col in FEATURE_COLUMNS:
    a, b = cic_data["X"][col].to_numpy(float), unsw_data["X"][col].to_numpy(float)
    ks = ks_2samp(np.log1p(np.clip(a, 0, None)), np.log1p(np.clip(b, 0, None)))
    rows.append({"variable": col, "ks_stat_log1p": ks.statistic,
                 "median_cic": np.median(a), "median_unsw": np.median(b),
                 "q95_cic": np.quantile(a, 0.95), "q95_unsw": np.quantile(b, 0.95)})
shift = pd.DataFrame(rows).round(4)
display(shift)
shift.to_csv(TABLES_DIR / "feature_shift_cic_vs_unsw.csv", index=False)

def overlap_report(src, tgt):
    ref = (src["X"].assign(_y=src["y"]).groupby(FEATURE_COLUMNS, as_index=False)["_y"].mean()
           .rename(columns={"_y": "_src_attack_rate"}))
    m = tgt["X"].assign(_y=tgt["y"]).merge(ref, on=FEATURE_COLUMNS, how="left")
    hit = m["_src_attack_rate"].notna()
    agree = ((m.loc[hit, "_src_attack_rate"] >= 0.5).astype(int) == m.loc[hit, "_y"]).mean() if hit.any() else np.nan
    return {"source": src["name"], "target": tgt["name"],
            "part_lignes_cible_avec_vecteur_identique_dans_source": round(float(hit.mean()), 4),
            "accord_des_labels_sur_ces_lignes": round(float(agree), 4) if hit.any() else np.nan}

display(pd.DataFrame([overlap_report(cic_data, unsw_data), overlap_report(unsw_data, cic_data)]))

,variable,ks_stat_log1p,median_cic,median_unsw,q95_cic,q95_unsw
0,flow_duration_s,0.2192,0.0313,0.0159,100.8287,1.6389
1,fwd_packets,0.4279,2.0000,12.0000,18.0000,122.0000
2,bwd_packets,0.4411,2.0000,12.0000,17.0000,126.0000
3,fwd_bytes,0.7001,62.0000,1470.0000,1767.0000,18890.0000
4,bwd_bytes,0.3550,122.0000,1832.0000,11601.0000,74204.1000


,source,target,part_lignes_cible_avec_vecteur_identique_dans_source,accord_des_labels_sur_ces_lignes
0,CIC-IDS2017,UNSW-NB15,0.0066,1.0
1,UNSW-NB15,CIC-IDS2017,0.0002,1.0


## 11. Interprétation et limites

1. Le F1 dépend du seuil ; PR-AUC et ROC-AUC n'en dépendent pas. Comparer F1 et PR-AUC sépare une perte de capacité de classement d'un mauvais calage du seuil source. La PR-AUC d'un classifieur aléatoire vaut la prévalence (`cross_prevalence`).
2. Les colonnes `*_diag_oracle_*` utilisent les labels de l'ensemble évalué : diagnostics, pas performances de détecteur.
3. Le seuil provient d'un seul split interne (logique v1). Sans `REFIT_ON_FULL_TRAIN`, le modèle est entraîné sur 80 % du train externe ; avec l'option, le seuil reste celui d'un modèle entraîné sur moins de données.
4. Le nombre de 90 expériences correspond à 90 entraînements ; chacun donne deux évaluations appariées (intra, cross). Les folds ne sont pas indépendants, et les **mêmes observations cibles sont réutilisées** dans toutes les évaluations cross : la dispersion cross est descriptive.
5. Les groupes sont des doublons exacts sur cinq variables, pas des sessions. Seules cinq variables communes sont utilisées : une chute cross-dataset peut refléter des définitions différentes (durée, direction, octets) ou des différences de labellisation.
6. Les avertissements de convergence du MLP sont comptés (`n_convergence_warnings`, `mlp_hit_max_iter`) ; `max_iter=300` ne garantit pas la convergence.
7. Résultats sur des échantillons de 100 000 lignes, pas sur les datasets complets.
8. Les fichiers `.npz` (scores, étiquettes vraies, indices, scores de validation) permettent l'ablation du seuil et l'analyse des erreurs sans réentraînement.

## 12. Analyses complémentaires des résultats

Cette section regroupe des analyses complémentaires destinées à approfondir
l'interprétation des résultats de l'évaluation cross-dataset.

### A. Synthèse des performances

Présentation des PR-AUC intra et cross, des prévalences cibles,
des ROC-AUC et des F1. Les résultats sont calculés sur les
90 expériences réalisées.

### B. Sensibilité du F1 au seuil de décision

Comparaison du F1 obtenu avec le seuil sélectionné sur la validation
interne et du F1 oracle calculé à titre strictement diagnostique sur
les observations évaluées.

Le F1 oracle n'est pas une mesure de performance généralisable :
il utilise les véritables étiquettes de l'ensemble évalué.

### C. PR-AUC pondérée par groupes

La PR-AUC est recalculée en attribuant à chaque observation un poids
inversement proportionnel à la taille de son groupe de vecteurs
de caractéristiques identiques.

Cette analyse mesure la sensibilité des résultats à la répétition
des mêmes vecteurs. Elle ne représente pas une évaluation sur des
sessions réseau indépendantes.

In [11]:
# --- A : tableau centré sur la PR-AUC ---
df = pd.read_csv(RESULTS_FILE)
df["cross_pr_auc_lift"] = df["cross_pr_auc"] / df["cross_prevalence"]
df["trivial_f1_all_attack"] = 2 * df["cross_prevalence"] / (1 + df["cross_prevalence"])
tab = df.groupby(["source", "target", "model"]).agg(
    intra_pr_auc=("intra_pr_auc", "mean"), cross_pr_auc=("cross_pr_auc", "mean"),
    prevalence_cible=("cross_prevalence", "mean"), lift_cross=("cross_pr_auc_lift", "mean"),
    intra_roc=("intra_roc_auc", "mean"), cross_roc=("cross_roc_auc", "mean"),
    cross_f1=("cross_f1", "mean"), cross_oracle_f1_DIAG=("cross_diag_oracle_f1", "mean"),
    f1_tout_attaque=("trivial_f1_all_attack", "mean"),
    taux_alertes_cross=("cross_pred_positive_rate", "mean")).round(4)
display(tab)
tab.to_csv(TABLES_DIR / "cross_dataset_pr_auc_table_v1_1.csv")

# --- B : écart seuil réel vs oracle (intra) + détail LR CIC -> UNSW ---
gap = df.groupby(["source", "model"]).agg(intra_f1=("intra_f1", "mean"), intra_f1_std=("intra_f1", "std"),
        intra_oracle_f1_DIAG=("intra_diag_oracle_f1", "mean")).round(4)
gap["ecart_seuil"] = (gap["intra_oracle_f1_DIAG"] - gap["intra_f1"]).round(4)
display(gap)
sub = df[(df.source == "CIC-IDS2017") & (df.model == "Logistic Regression")]
display(sub[["fold", "seed", "threshold", "cross_f1", "cross_pr_auc", "cross_pred_positive_rate",
             "cross_diag_oracle_f1"]].sort_values("cross_f1", ascending=False).round(4))

# --- C : PR-AUC cross pondérée par groupe (poids = 1 / taille du groupe de doublons) ---
def group_weights(data):
    counts = np.bincount(data["groups"])
    return 1.0 / counts[data["groups"]]
W = {"CIC-IDS2017": group_weights(cic_data), "UNSW-NB15": group_weights(unsw_data)}
rows = []
for r in df.itertuples():
    f = PRED_DIR / pred_filename(SHORT[r.source], SHORT[r.target], r.model, int(r.fold), int(r.seed))
    with np.load(f) as d:
        y, s = d["target_y_true"], d["target_score"]
    w = W[r.target]
    rows.append({"source": r.source, "target": r.target, "model": r.model,
                 "pr_auc_cross": r.cross_pr_auc,
                 "pr_auc_cross_ponderee": average_precision_score(y, s, sample_weight=w),
                 "prevalence": float(np.mean(y)), "prevalence_ponderee": float(np.average(y, weights=w))})
weighted = pd.DataFrame(rows).groupby(["source", "target", "model"]).mean().round(4)
display(weighted)
weighted.to_csv(TABLES_DIR / "cross_dataset_pr_auc_weighted_v1_1.csv")

intra_pr_auc  cross_pr_auc  prevalence_cible  lift_cross  intra_roc  cross_roc  cross_f1  cross_oracle_f1_DIAG  f1_tout_attaque  taux_alertes_cross
source      target      model                                                                                                                                                                   
CIC-IDS2017 UNSW-NB15   Logistic Regression        0.5157        0.6527            0.1258      5.1900     0.8303     0.8687    0.0533                0.7832           0.2234              0.0087
                        MLP                        0.8638        0.1338            0.1258      1.0641     0.9560     0.5298    0.0038                0.3330           0.2234              0.0052
                        Random Forest              0.9825        0.1346            0.1258      1.0702     0.9944     0.4654    0.0000                0.2234           0.2234              0.0000
UNSW-NB15   CIC-IDS2017 Logistic Regression        0.4074        0.4153            0.1979      2.0985     0.8408     0.7394    0.4474                0.4770           0.3304              0.4366
                        MLP                        0.9439        0.1996            0.1979      1.0086     0.9936     0.4784    0.1385                0.3755           0.3304              0.1539
                        Random Forest              0.9815        0.1871            0.1979      0.9455     0.9977     0.5063    0.1008                0.3689           0.3304              0.1802

intra_f1  intra_f1_std  intra_oracle_f1_DIAG  ecart_seuil
source      model                                                                         
CIC-IDS2017 Logistic Regression    0.6062        0.0350                0.6160       0.0098
            MLP                    0.7693        0.0337                0.7868       0.0175
            Random Forest          0.9536        0.0237                0.9548       0.0012
UNSW-NB15   Logistic Regression    0.4261        0.2216                0.6422       0.2161
            MLP                    0.6066        0.3365                0.9365       0.3299
            Random Forest          0.5732        0.3822                0.9617       0.3885

,fold,seed,threshold,cross_f1,cross_pr_auc,cross_pred_positive_rate,cross_diag_oracle_f1
11,4,2026,0.4802,0.7933,0.7298,0.0935,0.8568
5,2,2026,0.4749,0.0005,0.6474,0.0027,0.7780
1,1,123,0.4763,0.0005,0.6470,0.0027,0.7779
13,5,123,0.4774,0.0005,0.6472,0.0027,0.7777
12,5,42,0.4758,0.0005,0.6472,0.0027,0.7777
14,5,2026,0.4764,0.0005,0.6472,0.0027,0.7777
0,1,42,0.4736,0.0005,0.6470,0.0027,0.7779
6,3,42,0.4727,0.0005,0.6469,0.0027,0.7780
4,2,123,0.4699,0.0005,0.6474,0.0027,0.7781
3,2,42,0.4687,0.0005,0.6475,0.0027,0.7781


pr_auc_cross  pr_auc_cross_ponderee  prevalence  prevalence_ponderee
source      target      model                                                                                    
CIC-IDS2017 UNSW-NB15   Logistic Regression        0.6527                 0.0850      0.1258               0.0421
                        MLP                        0.1338                 0.0650      0.1258               0.0421
                        Random Forest              0.1346                 0.0696      0.1258               0.0421
UNSW-NB15   CIC-IDS2017 Logistic Regression        0.4153                 0.4864      0.1979               0.1637
                        MLP                        0.1996                 0.1580      0.1979               0.1637
                        Random Forest              0.1871                 0.1561      0.1979               0.1637

### D. Comparaison des lifts

Le lift correspond au rapport entre la PR-AUC et la prévalence
de la classe positive.

Il est présenté pour les évaluations par lignes et par motifs.
Sa comparaison doit tenir compte de la prévalence et de son maximum
théorique, égal à l'inverse de celle-ci.

In [12]:
# --- D : lifts (PR-AUC / prévalence), lignes vs motifs distincts, intra ET cross ---
rows = []
for r in df.itertuples():
    f = PRED_DIR / pred_filename(SHORT[r.source], SHORT[r.target], r.model, int(r.fold), int(r.seed))
    with np.load(f) as d:
        ys, ss, idx = d["source_test_y_true"], d["source_test_score"], d["source_test_idx"]
        yt, st = d["target_y_true"], d["target_score"]
    ws, wt = W[r.source][idx], W[r.target]
    pi = average_precision_score(ys, ss, sample_weight=ws)
    pc = average_precision_score(yt, st, sample_weight=wt)
    rows.append({"source": r.source, "target": r.target, "model": r.model,
                 "intra_lift_lignes": r.intra_pr_auc / r.intra_prevalence,
                 "intra_lift_motifs": pi / float(np.average(ys, weights=ws)),
                 "cross_lift_lignes": r.cross_pr_auc / r.cross_prevalence,
                 "cross_lift_motifs": pc / float(np.average(yt, weights=wt))})
lifts = pd.DataFrame(rows).groupby(["source", "target", "model"]).agg(["mean", "std"]).round(3)
display(lifts)
lifts.to_csv(TABLES_DIR / "cross_dataset_lifts_v1_1.csv")

intra_lift_lignes        intra_lift_motifs         cross_lift_lignes        cross_lift_motifs       
                                                         mean    std              mean     std              mean    std              mean    std
source      target      model                                                                                                                   
CIC-IDS2017 UNSW-NB15   Logistic Regression             2.606  0.203             3.240   0.340             5.190  0.169             2.017  0.144
                        MLP                             4.365  0.153             5.858   0.062             1.064  0.150             1.543  0.404
                        Random Forest                   4.965  0.049             6.064   0.010             1.070  0.156             1.651  0.239
UNSW-NB15   CIC-IDS2017 Logistic Regression             3.240  0.726             4.835   1.537             2.099  0.145             2.971  0.264
                        MLP                             7.505  0.240            28.120  25.157             1.009  0.129             0.965  0.194
                        Random Forest                   7.804  0.081            36.919  39.396             0.946  0.091             0.953  0.107

### E. Gain normalisé

Le gain normalisé est calculé selon la formule :

G = (PR-AUC - prévalence) / (1 - prévalence)

Il permet de rapporter la PR-AUC à sa référence de prévalence.
Il est calculé par expérience, puis résumé par moyenne et écart-type.

Les résultats sont présentés par lignes et par motifs distincts.

### F. Composition des folds externes

Cette analyse décrit le nombre de lignes, de groupes et de groupes
contenant des attaques dans chaque fold externe.

Elle permet d'identifier d'éventuelles concentrations de vecteurs
identiques et des différences de composition entre les folds.

Les groupes sont définis par l'identité exacte des cinq caractéristiques
communes et ne correspondent pas nécessairement à des sessions réseau.

In [13]:
# --- E : gain normalisé (0 = hasard, 1 = parfait), lignes vs motifs distincts ---
def ngain(pr_auc, prev):
    return (pr_auc - prev) / (1 - prev)

rows = []
for r in df.itertuples():
    f = PRED_DIR / pred_filename(SHORT[r.source], SHORT[r.target], r.model, int(r.fold), int(r.seed))
    with np.load(f) as d:
        ys, ss, idx = d["source_test_y_true"], d["source_test_score"], d["source_test_idx"]
        yt, st = d["target_y_true"], d["target_score"]
    ws, wt = W[r.source][idx], W[r.target]
    pi = average_precision_score(ys, ss, sample_weight=ws)
    pc = average_precision_score(yt, st, sample_weight=wt)
    rows.append({"source": r.source, "target": r.target, "model": r.model,
                 "intra_gain_lignes": ngain(r.intra_pr_auc, r.intra_prevalence),
                 "intra_gain_motifs": ngain(pi, float(np.average(ys, weights=ws))),
                 "cross_gain_lignes": ngain(r.cross_pr_auc, r.cross_prevalence),
                 "cross_gain_motifs": ngain(pc, float(np.average(yt, weights=wt)))})
gains = pd.DataFrame(rows).groupby(["source", "target", "model"]).agg(["mean", "std"]).round(3)
display(gains)
gains.to_csv(TABLES_DIR / "cross_dataset_normalized_gains_v1_1.csv")

# --- F : composition des folds externes ---
rows = []
for data in (cic_data, unsw_data):
    sizes = np.bincount(data["groups"])
    for k, (tr, te) in enumerate(data["outer_splits"], 1):
        g, y = data["groups"][te], data["y"][te]
        rows.append({"dataset": data["name"], "fold": k, "lignes_test": len(te),
                     "groupes_test": len(np.unique(g)), "lignes_attaque": int(y.sum()),
                     "groupes_avec_attaque": len(np.unique(g[y == 1])),
                     "plus_gros_groupe": int(sizes[g].max()),
                     "prevalence_lignes": round(float(y.mean()), 4),
                     "prevalence_motifs": round(float(np.average(y, weights=1.0 / sizes[g])), 4)})
display(pd.DataFrame(rows))

intra_gain_lignes        intra_gain_motifs        cross_gain_lignes        cross_gain_motifs       
                                                         mean    std              mean    std              mean    std              mean    std
source      target      model                                                                                                                  
CIC-IDS2017 UNSW-NB15   Logistic Regression             0.396  0.050             0.438  0.066             0.603  0.024             0.045  0.006
                        MLP                             0.830  0.038             0.951  0.011             0.009  0.022             0.024  0.018
                        Random Forest                   0.978  0.012             0.991  0.002             0.010  0.022             0.029  0.010
UNSW-NB15   CIC-IDS2017 Logistic Regression             0.322  0.104             0.139  0.061             0.271  0.036             0.386  0.052
                        MLP                             0.936  0.034             0.704  0.227             0.002  0.032            -0.007  0.038
                        Random Forest                   0.979  0.011             0.816  0.219            -0.013  0.023            -0.009  0.021

,dataset,fold,lignes_test,groupes_test,lignes_attaque,groupes_avec_attaque,plus_gros_groupe,prevalence_lignes,prevalence_motifs
0,CIC-IDS2017,1,20001,14858,3958,2464,469,0.1979,0.1637
1,CIC-IDS2017,2,19999,14862,3957,2462,659,0.1979,0.1639
2,CIC-IDS2017,3,19999,14867,3957,2463,266,0.1979,0.1639
3,CIC-IDS2017,4,19999,14834,3957,2443,1180,0.1979,0.1631
4,CIC-IDS2017,5,20002,14873,3959,2468,286,0.1979,0.1638
5,UNSW-NB15,1,19996,12342,2513,45,2469,0.1257,0.0035
6,UNSW-NB15,2,20003,12948,2517,676,1814,0.1258,0.0516
7,UNSW-NB15,3,19997,12935,2514,664,1198,0.1257,0.0509
8,UNSW-NB15,4,20009,12927,2520,663,1125,0.1259,0.0508
9,UNSW-NB15,5,19995,12965,2513,678,1069,0.1257,0.0520



## 13. Synthèse des analyses complémentaires

Les analyses complémentaires mettent en évidence une forte asymétrie
de généralisation entre CIC-IDS2017 et UNSW-NB15.

La régression logistique conserve un gain normalisé positif dans les
deux directions de transfert, tandis que les gains du MLP et du
Random Forest deviennent proches de zéro en évaluation cross-dataset.

La comparaison entre les métriques calculées par lignes et celles
pondérées par motifs révèle une sensibilité aux répétitions de
vecteurs identiques, particulièrement dans le transfert
CIC-IDS2017 vers UNSW-NB15.

L'analyse de la composition des folds révèle également une
concentration importante des observations d'attaque dans certains
groupes d'UNSW-NB15, particulièrement dans le premier fold.

Ces résultats apportent des éléments descriptifs pour comprendre
les difficultés de transfert. Ils ne permettent cependant pas
d'attribuer causalement la dégradation des performances à des
artefacts ou à une caractéristique particulière.

Une analyse par ablation des variables sera nécessaire pour
approfondir cette question.
